# Null Imputation Prep

Before imputing missing values, this notebook checks whether columns are related to each other across mixed data types.

In [1]:
import itertools

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

## Load Data

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)

train shape: (690088, 15)
test shape: (295753, 14)


## Missing Value Summary

In [3]:
null_summary = (
    pd.DataFrame({
        "dtype": train.dtypes.astype(str),
        "count": train.isna().sum(),
        "percent": train.isna().mean().mul(100),
        "nunique": train.nunique(dropna=True),
    })
    .sort_values("percent", ascending=False)
)

null_summary.style.format({"percent": "{:.2f}"})

,dtype,count,percent,nunique
stress_level,str,82811,12.00,3
sleep_duration,float64,75999,11.01,701
sleep_quality,str,58331,8.45,3
calorie_expenditure,float64,52853,7.66,2101
water_intake,float64,43477,6.30,400
physical_activity_level,str,36621,5.31,3
smoking_alcohol,str,28582,4.14,3
gender,str,21373,3.10,3
step_count,float64,13916,2.02,12807
bmi,float64,13898,2.01,1596


In [4]:
feature_cols = [col for col in train.columns if col not in ["id", "health_condition"]]
numeric_cols = train[feature_cols].select_dtypes(include="number").columns.tolist()
categorical_cols = train[feature_cols].select_dtypes(exclude="number").columns.tolist()
columns_for_dependency = [col for col in train.columns if col != "id"]

print("numeric feature columns:", numeric_cols)
print("categorical feature columns:", categorical_cols)
print("dependency check columns:", columns_for_dependency)

numeric feature columns: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
categorical feature columns: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
dependency check columns: ['health_condition', 'sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


## Mixed-Type Column Dependency Check

The score is always between 0 and 1, where higher means a stronger relationship.

- Numeric vs numeric: absolute Spearman correlation
- Categorical vs categorical: bias-corrected Cramer's V
- Numeric vs categorical: correlation ratio

In [5]:
def _valid_pair(left, right):
    valid = left.notna() & right.notna()
    return left[valid], right[valid]


def is_numeric(series):
    return pd.api.types.is_numeric_dtype(series)


def cramers_v(left, right):
    left, right = _valid_pair(left, right)
    if left.nunique(dropna=True) < 2 or right.nunique(dropna=True) < 2:
        return np.nan

    observed = pd.crosstab(left, right)
    if observed.shape[0] < 2 or observed.shape[1] < 2:
        return np.nan

    observed_values = observed.to_numpy(dtype=float)
    n = observed_values.sum()
    row_sum = observed_values.sum(axis=1, keepdims=True)
    col_sum = observed_values.sum(axis=0, keepdims=True)
    expected = row_sum @ col_sum / n

    chi2 = np.divide(
        (observed_values - expected) ** 2,
        expected,
        out=np.zeros_like(expected),
        where=expected != 0,
    ).sum()

    phi2 = chi2 / n
    rows, cols = observed.shape

    # Bias correction keeps high-cardinality categorical pairs from being overstated.
    phi2_corr = max(0, phi2 - ((cols - 1) * (rows - 1)) / (n - 1))
    rows_corr = rows - ((rows - 1) ** 2) / (n - 1)
    cols_corr = cols - ((cols - 1) ** 2) / (n - 1)
    denominator = min(cols_corr - 1, rows_corr - 1)

    if denominator <= 0:
        return np.nan
    return np.sqrt(phi2_corr / denominator)


def correlation_ratio(categories, values):
    categories, values = _valid_pair(categories, pd.to_numeric(values, errors="coerce"))
    valid_values = values.notna()
    categories = categories[valid_values]
    values = values[valid_values].astype(float)

    if categories.nunique(dropna=True) < 2 or values.nunique(dropna=True) < 2:
        return np.nan

    grouped = values.groupby(categories)
    counts = grouped.count()
    means = grouped.mean()
    overall_mean = values.mean()

    between_group_ss = (counts * (means - overall_mean) ** 2).sum()
    total_ss = ((values - overall_mean) ** 2).sum()

    if total_ss == 0:
        return np.nan
    return np.sqrt(between_group_ss / total_ss)


def association_score(left, right):
    left_is_numeric = is_numeric(left)
    right_is_numeric = is_numeric(right)

    if left_is_numeric and right_is_numeric:
        left, right = _valid_pair(left, right)
        if left.nunique(dropna=True) < 2 or right.nunique(dropna=True) < 2:
            return np.nan
        return abs(left.corr(right, method="spearman"))

    if not left_is_numeric and not right_is_numeric:
        return cramers_v(left, right)

    if left_is_numeric:
        return correlation_ratio(right, left)
    return correlation_ratio(left, right)


def column_kind(series):
    return "numeric" if is_numeric(series) else "categorical"

In [6]:
# Association rules are learned from train only. Test is not used here.
dependency_sample = train[columns_for_dependency].copy()

print("Rows used for train-only dependency checks:", len(dependency_sample))

Rows used for train-only dependency checks: 690088


In [7]:
dependency_rows = []

for col_1, col_2 in itertools.combinations(columns_for_dependency, 2):
    score = association_score(dependency_sample[col_1], dependency_sample[col_2])
    dependency_rows.append({
        "column_1": col_1,
        "column_2": col_2,
        "type_1": column_kind(dependency_sample[col_1]),
        "type_2": column_kind(dependency_sample[col_2]),
        "association": score,
        "non_null_pairs": int((dependency_sample[col_1].notna() & dependency_sample[col_2].notna()).sum()),
    })

dependency_summary = (
    pd.DataFrame(dependency_rows)
    .sort_values("association", ascending=False, na_position="last")
    .reset_index(drop=True)
)

dependency_summary.head(50).style.format({"association": "{:.3f}"})

,column_1,column_2,type_1,type_2,association,non_null_pairs
0,step_count,physical_activity_level,numeric,categorical,0.664,640288
1,exercise_duration,physical_activity_level,numeric,categorical,0.658,646936
2,step_count,exercise_duration,numeric,numeric,0.441,669402
3,health_condition,sleep_duration,categorical,numeric,0.437,614089
4,health_condition,stress_level,categorical,categorical,0.412,607277
5,sleep_duration,sleep_quality,numeric,categorical,0.376,562161
6,calorie_expenditure,physical_activity_level,numeric,categorical,0.372,603379
7,calorie_expenditure,exercise_duration,numeric,numeric,0.370,630836
8,calorie_expenditure,step_count,numeric,numeric,0.367,624390
9,health_condition,physical_activity_level,categorical,categorical,0.241,653467


In [8]:
association_matrix = pd.DataFrame(
    np.eye(len(columns_for_dependency)),
    index=columns_for_dependency,
    columns=columns_for_dependency,
)

for row in dependency_summary.itertuples(index=False):
    association_matrix.loc[row.column_1, row.column_2] = row.association
    association_matrix.loc[row.column_2, row.column_1] = row.association

association_matrix.style.background_gradient(cmap="viridis", axis=None).format("{:.2f}")

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
health_condition,1.00,0.44,0.01,0.17,0.10,0.19,0.19,0.00,0.02,0.41,0.12,0.24,0.08,0.01
sleep_duration,0.44,1.00,0.00,0.06,0.00,0.00,0.00,0.00,0.01,0.08,0.38,0.01,0.04,0.00
heart_rate,0.01,0.00,1.00,0.00,0.00,0.01,0.01,0.00,0.01,0.01,0.00,0.01,0.00,0.00
bmi,0.17,0.06,0.00,1.00,0.11,0.02,0.02,0.00,0.00,0.08,0.03,0.04,0.02,0.00
calorie_expenditure,0.10,0.00,0.00,0.11,1.00,0.37,0.37,0.00,0.00,0.01,0.00,0.37,0.01,0.00
step_count,0.19,0.00,0.01,0.02,0.37,1.00,0.44,0.00,0.01,0.02,0.00,0.66,0.01,0.00
exercise_duration,0.19,0.00,0.01,0.02,0.37,0.44,1.00,0.00,0.01,0.02,0.00,0.66,0.01,0.00
water_intake,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.01,0.01,0.00,0.01,0.00,0.01
diet_type,0.02,0.01,0.01,0.00,0.00,0.01,0.01,0.01,1.00,0.01,0.00,0.01,0.00,0.01
stress_level,0.41,0.08,0.01,0.08,0.01,0.02,0.02,0.01,0.01,1.00,0.02,0.02,0.04,0.01


In [9]:
forward_relationships = dependency_summary.rename(columns={
    "column_1": "column",
    "column_2": "related_column",
    "type_1": "column_type",
    "type_2": "related_column_type",
})
reverse_relationships = dependency_summary.rename(columns={
    "column_2": "column",
    "column_1": "related_column",
    "type_2": "column_type",
    "type_1": "related_column_type",
})

strongest_relationships_by_column = (
    pd.concat([forward_relationships, reverse_relationships], ignore_index=True)
    [["column", "related_column", "column_type", "related_column_type", "association", "non_null_pairs"]]
    .sort_values(["column", "association"], ascending=[True, False])
    .groupby("column", as_index=False)
    .head(5)
    .reset_index(drop=True)
)

strongest_relationships_by_column.style.format({"association": "{:.3f}"})

,column,related_column,column_type,related_column_type,association,non_null_pairs
0,bmi,health_condition,numeric,categorical,0.174,676190
1,bmi,calorie_expenditure,numeric,numeric,0.110,624389
2,bmi,stress_level,numeric,categorical,0.083,595057
3,bmi,sleep_duration,numeric,numeric,0.061,601658
4,bmi,physical_activity_level,numeric,categorical,0.037,640380
5,calorie_expenditure,physical_activity_level,numeric,categorical,0.372,603379
6,calorie_expenditure,exercise_duration,numeric,numeric,0.370,630836
7,calorie_expenditure,step_count,numeric,numeric,0.367,624390
8,calorie_expenditure,bmi,numeric,numeric,0.110,624389
9,calorie_expenditure,health_condition,numeric,categorical,0.101,637235


## Missingness Dependency Check

This checks whether the fact that a column is missing is related to the other column values. This is useful before imputation because missingness can itself be systematic.

In [10]:
nullable_cols = null_summary.query("count > 0").index.tolist()
missingness_rows = []

for missing_col in nullable_cols:
    missing_flag = dependency_sample[missing_col].isna().map({True: "missing", False: "present"})

    for predictor_col in columns_for_dependency:
        if predictor_col == missing_col:
            continue

        score = association_score(missing_flag, dependency_sample[predictor_col])
        missingness_rows.append({
            "missing_column": missing_col,
            "predictor_column": predictor_col,
            "predictor_type": column_kind(dependency_sample[predictor_col]),
            "missing_pct": dependency_sample[missing_col].isna().mean() * 100,
            "association_with_missingness": score,
        })

missingness_dependency = (
    pd.DataFrame(missingness_rows)
    .sort_values("association_with_missingness", ascending=False, na_position="last")
    .reset_index(drop=True)
)

missingness_dependency.head(60).style.format({
    "missing_pct": "{:.2f}",
    "association_with_missingness": "{:.3f}",
})

,missing_column,predictor_column,predictor_type,missing_pct,association_with_missingness
0,water_intake,gender,categorical,6.30,0.372
1,physical_activity_level,smoking_alcohol,categorical,5.31,0.338
2,bmi,sleep_quality,categorical,2.01,0.206
3,diet_type,gender,categorical,1.00,0.145
4,bmi,sleep_duration,numeric,2.01,0.067
5,bmi,health_condition,categorical,2.01,0.031
6,bmi,smoking_alcohol,categorical,2.01,0.006
7,bmi,stress_level,categorical,2.01,0.006
8,water_intake,diet_type,categorical,6.30,0.004
9,water_intake,stress_level,categorical,6.30,0.004


In [11]:
top_missingness_predictors = (
    missingness_dependency
    .sort_values(["missing_column", "association_with_missingness"], ascending=[True, False])
    .groupby("missing_column", as_index=False)
    .head(5)
    .reset_index(drop=True)
)

top_missingness_predictors.style.format({
    "missing_pct": "{:.2f}",
    "association_with_missingness": "{:.3f}",
})

,missing_column,predictor_column,predictor_type,missing_pct,association_with_missingness
0,bmi,sleep_quality,categorical,2.01,0.206
1,bmi,sleep_duration,numeric,2.01,0.067
2,bmi,health_condition,categorical,2.01,0.031
3,bmi,smoking_alcohol,categorical,2.01,0.006
4,bmi,stress_level,categorical,2.01,0.006
5,calorie_expenditure,bmi,numeric,7.66,0.003
6,calorie_expenditure,water_intake,numeric,7.66,0.002
7,calorie_expenditure,exercise_duration,numeric,7.66,0.002
8,calorie_expenditure,stress_level,categorical,7.66,0.001
9,calorie_expenditure,diet_type,categorical,7.66,0.001


## Association-Based Imputation

Association is now used in the imputation step. For each feature with missing values, the notebook uses train-only associations to find predictor columns with association >= `0.40`.

The process is:

1. Fit simple median/mode imputers on train and apply them to train/test as a fallback.
2. For columns with strong associated predictors, train a model on train rows where the target column is known.
3. Use that fitted train-only model to fill missing values in both train and test.

`id` and `health_condition` are not used as imputation predictors.

In [12]:
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ASSOCIATION_THRESHOLD = 0.40
MIN_KNOWN_ROWS_FOR_MODEL = 1_000
MODEL_RANDOM_STATE = 42

train_imputed = train.copy()
test_imputed = test.copy()

# Stage 1: train-fitted fallback imputation. These values are also used as clean model inputs.
numeric_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

if numeric_cols:
    train_imputed[numeric_cols] = numeric_imputer.fit_transform(train[numeric_cols])
    test_imputed[numeric_cols] = numeric_imputer.transform(test[numeric_cols])

if categorical_cols:
    train_imputed[categorical_cols] = categorical_imputer.fit_transform(train[categorical_cols])
    test_imputed[categorical_cols] = categorical_imputer.transform(test[categorical_cols])


def get_strong_predictors(target_col, threshold=ASSOCIATION_THRESHOLD):
    predictors = []

    for predictor_col in feature_cols:
        if predictor_col == target_col:
            continue

        score = association_matrix.loc[target_col, predictor_col]
        if pd.notna(score) and score >= threshold:
            predictors.append((predictor_col, score))

    return sorted(predictors, key=lambda item: item[1], reverse=True)


def build_model_imputer(target_col, predictors):
    predictor_numeric_cols = [col for col in predictors if col in numeric_cols]
    predictor_categorical_cols = [col for col in predictors if col in categorical_cols]
    transformers = []

    if predictor_numeric_cols:
        transformers.append((
            "numeric",
            SimpleImputer(strategy="median"),
            predictor_numeric_cols,
        ))

    if predictor_categorical_cols:
        transformers.append((
            "categorical",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            predictor_categorical_cols,
        ))

    preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

    if target_col in numeric_cols:
        estimator = HistGradientBoostingRegressor(random_state=MODEL_RANDOM_STATE)
    else:
        estimator = HistGradientBoostingClassifier(random_state=MODEL_RANDOM_STATE)

    return Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", estimator),
    ])


def predict_and_clip(model, target_col, frame):
    predictions = model.predict(frame)

    if target_col in numeric_cols:
        known_values = train[target_col].dropna()
        predictions = np.clip(predictions, known_values.min(), known_values.max())

    return predictions


model_reports = []
imputation_plan_rows = []
imputation_targets = [
    col for col in feature_cols
    if train[col].isna().any() or test[col].isna().any()
]

for target_col in imputation_targets:
    strong_predictors = get_strong_predictors(target_col)
    predictor_cols = [col for col, _ in strong_predictors]
    known_mask = train[target_col].notna()
    train_missing_mask = train[target_col].isna()
    test_missing_mask = test[target_col].isna()

    imputation_plan_rows.append({
        "target_column": target_col,
        "target_type": column_kind(train[target_col]),
        "train_missing": int(train_missing_mask.sum()),
        "test_missing": int(test_missing_mask.sum()),
        "method": "model" if predictor_cols and known_mask.sum() >= MIN_KNOWN_ROWS_FOR_MODEL else "median/mode fallback",
        "predictors_from_train_association": ", ".join(
            f"{col} ({score:.3f})" for col, score in strong_predictors
        ),
    })

    if not predictor_cols or known_mask.sum() < MIN_KNOWN_ROWS_FOR_MODEL:
        continue

    model = build_model_imputer(target_col, predictor_cols)
    model.fit(train_imputed.loc[known_mask, predictor_cols], train.loc[known_mask, target_col])

    if train_missing_mask.any():
        train_imputed.loc[train_missing_mask, target_col] = predict_and_clip(
            model,
            target_col,
            train_imputed.loc[train_missing_mask, predictor_cols],
        )

    if test_missing_mask.any():
        test_imputed.loc[test_missing_mask, target_col] = predict_and_clip(
            model,
            target_col,
            test_imputed.loc[test_missing_mask, predictor_cols],
        )

    model_reports.append({
        "target_column": target_col,
        "model_type": "regressor" if target_col in numeric_cols else "classifier",
        "predictors": predictor_cols,
        "known_train_rows": int(known_mask.sum()),
        "filled_train_rows": int(train_missing_mask.sum()),
        "filled_test_rows": int(test_missing_mask.sum()),
    })

imputation_plan = pd.DataFrame(imputation_plan_rows)
model_imputation_report = pd.DataFrame(model_reports)

print(f"Association threshold used for model imputation: {ASSOCIATION_THRESHOLD}")
display(imputation_plan)

print("Model imputers trained on train and applied to train/test:")
display(model_imputation_report)

print("Remaining missing values in train features:", int(train_imputed[feature_cols].isna().sum().sum()))
print("Remaining missing values in test features:", int(test_imputed[feature_cols].isna().sum().sum()))

Association threshold used for model imputation: 0.4


,target_column,target_type,train_missing,test_missing,method,predictors_from_train_association
0,sleep_duration,numeric,75999,32571,median/mode fallback,
1,heart_rate,numeric,7833,3357,median/mode fallback,
2,bmi,numeric,13898,5956,median/mode fallback,
3,calorie_expenditure,numeric,52853,22652,median/mode fallback,
4,step_count,numeric,13916,5964,model,"physical_activity_level (0.664), exercise_duration (0.441)"
5,exercise_duration,numeric,6901,2958,model,"physical_activity_level (0.658), step_count (0.441)"
6,water_intake,numeric,43477,18633,median/mode fallback,
7,diet_type,categorical,6901,2958,median/mode fallback,
8,stress_level,categorical,82811,35490,median/mode fallback,
9,sleep_quality,categorical,58331,24999,median/mode fallback,


Model imputers trained on train and applied to train/test:


,target_column,model_type,predictors,known_train_rows,filled_train_rows,filled_test_rows
0,step_count,regressor,"[physical_activity_level, exercise_duration]",676172,13916,5964
1,exercise_duration,regressor,"[physical_activity_level, step_count]",683187,6901,2958
2,physical_activity_level,classifier,"[step_count, exercise_duration]",653467,36621,15695


Remaining missing values in train features: 0
Remaining missing values in test features: 0


In [13]:
train_imputed.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,medium,average,sedentary,yes,male


In [18]:
test_imputed.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,6.99,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,male
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


In [19]:
train_imputed.isna().sum().sum(), test_imputed.isna().sum().sum()

(np.int64(0), np.int64(0))

In [20]:
train_imputed.to_csv("data/train_imputed.csv", index=False)
test_imputed.to_csv("data/test_imputed.csv", index=False)